In [2]:
import os
import time
import math
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
from matplotlib import rc
rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta

import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from datetime import date

from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
import statsmodels.api as sm

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


### 데이터 전처리

In [ ]:
# # 코랩에서 실행할 경우 해당 셀 실행
# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# file_path = '/content/drive/Shareddrives/TD MKT/99. 데이터 분석/수거신청 물량 수요 예측/'

## 모델 - Prophet


In [9]:
def prophet_fitting(data, change_point_prior_scale) :
    pp = Prophet()

    holidays1 = pd.DataFrame({
        'holiday': '대체',
        'ds' : ['2022-03-09', '2022-05-09', '2022-06-01', '2022-09-12'
                , '2023-05-29', '2023-10-02'
                , '2024-04-10', '2024-05-06'
                , '2025-03-03', '2025-05-06', '2025-06-03', '2025-10-08'
                , '2026-03-02', '2026-05-25', '2026-08-17', '2026-10-05'],
        'lower_window': -1, 
        'upper_window': 1,
        'changepoint_prior_scale' : change_point_prior_scale
    })
    final_holidays = pd.concat([pp.holidays,  holidays1]).drop_duplicates(subset='ds')
    pp.add_seasonality(name = 'weekly', period = 7, fourier_order = 5)
    pp.add_seasonality(name = 'monthly', period = 30.5, fourier_order = 5)
    # pp.add_regressor(step_column_name)
    pp.fit(data)
    
    future_dates = pd.date_range(start= date.today().strftime('%Y-%m-%d'), end="2026-06-30", freq="D")
    future_df = pd.DataFrame({"ds": future_dates})
    
    forecast = pp.predict(future_df)
    forecast.index = forecast.ds
    pred = forecast['yhat'].astype(int)
    return forecast

path = os.getcwd()
df = pd.read_excel('new_user.xlsx').dropna(axis = 0)
df_subs = df[['ds', 'subs']]
df_free = df[['ds', 'free']]

df_subs.columns = ['ds', 'y']
df_free.columns = ['ds', 'y']

# df_subs
pred_subs = prophet_fitting(df_subs,  0.1)
pred_free = prophet_fitting(df_free,  0.1)
tmp = pred_subs['yhat'] + pred_free['yhat']


x = df['y']
y = df['sum_paid']
model = sm.OLS(y, x)
result = model.fit()

result.summary()

pred_list = result.predict(list(tmp)).astype(int)
date_list = pd.DataFrame(pd.date_range(start = date.today().strftime('%Y-%m-%d'), end = '2026-06-30', freq = 'D'))
maechul1 = pd.DataFrame(pred_list)
maechul = pd.concat([date_list, maechul1], axis = 1)
maechul.columns = ['date', 'sales']
total = pd.merge(maechul ,pd.DataFrame(tmp).reset_index(), left_on = 'date', right_on = 'ds')[['date', 'yhat', 'sales']]
total['yhat'] =  total['yhat'].astype('int')
total.to_excel('2026년 상반기 예상 수거신청 및 매출.xlsx')
total

14:16:15 - cmdstanpy - INFO - Chain [1] start processing
14:16:15 - cmdstanpy - INFO - Chain [1] done processing
14:16:15 - cmdstanpy - INFO - Chain [1] start processing
14:16:15 - cmdstanpy - INFO - Chain [1] done processing


,date,yhat,sales
0,2026-02-02,2858,94135643
1,2026-02-03,2499,82313117
2,2026-02-04,2452,80769455
3,2026-02-05,2415,79553849
4,2026-02-06,3004,98933510
...,...,...,...
144,2026-06-26,3410,112295584
145,2026-06-27,3735,122992623
146,2026-06-28,4111,135376927
147,2026-06-29,3183,104820016
